# Task 1: Data Collection and Preprocessing

**Project:** Customer Experience Analytics for Fintech Apps  
**Week:** 10 Academy KAIM9 – Week 2  
**Objective:** Scrape Google Play Store reviews for three Ethiopian banks, clean and preprocess the data, and save it as a structured CSV ready for analysis.

**Banks:**
- Commercial Bank of Ethiopia (CBE)
- Bank of Abyssinia (BOA)
- Dashen Bank

**Target:** 400+ reviews per bank (1,200+ total)

## 1. Install & Import Libraries

In [3]:
import sys
!{sys.executable} -m pip install google-play-scraper pandas

  Using cached google_play_scraper-1.2.7-py3-none-any.whl.metadata (50 kB)
Using cached google_play_scraper-1.2.7-py3-none-any.whl (28 kB)


In [2]:
!pip install google-play-scraper pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import os
import time
import logging
from datetime import datetime
from google_play_scraper import reviews, Sort

# Set up logging so we can track progress and issues
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 2. Define App Configuration

Each bank's app ID was obtained from their Google Play Store URL.  
Format: `https://play.google.com/store/apps/details?id=<APP_ID>`

In [35]:
# App configuration for each bank
BANK_APPS = [
    {
        'bank': 'Commercial Bank of Ethiopia',
        'app_id': 'com.combanketh.mobilebanking',
        'app_name': 'CBE Mobile'
    },
    {
        'bank': 'Bank of Abyssinia',
        'app_id': 'com.boa.boaMobileBanking',
        'app_name': 'BOA Mobile'
    },
    {
        'bank': 'Dashen Bank',
        'app_id': 'com.dashen.dashensuperapp',
        'app_name': 'Dashen Super App'
    }
]

# Scraping settings
TARGET_REVIEWS_PER_BANK = 600  # Minimum required
REVIEWS_PER_BATCH       = 200   # Max per API call (library limit)
LANGUAGE                = 'en'
COUNTRY                 = 'et'  # Ethiopia

print(f"✅ Configuration set: targeting {TARGET_REVIEWS_PER_BANK}+ reviews per bank")
print(f"   Banks to scrape: {[b['bank'] for b in BANK_APPS]}")

✅ Configuration set: targeting 600+ reviews per bank
   Banks to scrape: ['Commercial Bank of Ethiopia', 'Bank of Abyssinia', 'Dashen Bank']


## 3. Scraping Function

We use `google-play-scraper` with pagination (continuation tokens) to collect enough reviews.  
Reviews are sorted by **newest first** to get the most relevant data.

In [36]:
def scrape_reviews(app_id, bank_name, target_count=400, lang='en', country='et'):
    """
    Scrape reviews from Google Play Store for a given app.

    Parameters:
        app_id       (str): Google Play app package name
        bank_name    (str): Human-readable bank name for labeling
        target_count (int): Minimum number of reviews to collect
        lang         (str): Language code for reviews
        country      (str): Country code

    Returns:
        list[dict]: Raw review records
    """
    all_reviews = []
    continuation_token = None
    batch_num = 0

    logger.info(f"Starting scrape for {bank_name} (app_id: {app_id})")

    while len(all_reviews) < target_count:
        try:
            batch_num += 1
            result, continuation_token = reviews(
                app_id,
                lang=lang,
                country=country,
                sort=Sort.NEWEST,
                count=200,
                continuation_token=continuation_token
            )

            if not result:
                logger.warning(f"  No more reviews returned for {bank_name} at batch {batch_num}")
                break

            all_reviews.extend(result)
            logger.info(f"  Batch {batch_num}: fetched {len(result)} reviews | Total so far: {len(all_reviews)}")

            # Stop if no continuation token (no more pages)
            if continuation_token is None:
                logger.info(f"  No more pages available for {bank_name}")
                break

            # Polite delay between batches to avoid rate limiting
            time.sleep(2)

        except Exception as e:
            logger.error(f"  Error scraping {bank_name} at batch {batch_num}: {e}")
            # Wait longer and retry once on error
            time.sleep(5)
            break

    logger.info(f"✅ Done scraping {bank_name}: {len(all_reviews)} total reviews collected")
    return all_reviews


print("✅ Scraping function defined")

✅ Scraping function defined


In [37]:
from google_play_scraper import search

results = search(
    'Dashen Bank Ethiopia',
    lang='en',
    country='et',
    n_hits=5
)

for r in results:
    print(r['appId'], '|', r['title'])

com.dashen.dashensuperapp | Dashen Bank
com.cr2.amolelight | Dashen Mobile
com.dashen.dashenmerchant | Dashen Bank Merchant
com.combanketh.mobilebanking | Commercial Bank of Ethiopia
com.amole.agent | MobilePlus Biz


## 4. Run the Scraper for All Three Banks

> ⏳ This cell may take 2–5 minutes to complete. Each bank is scraped separately with a delay between banks to avoid rate limiting.

In [38]:
raw_data = {}  # Store raw results per bank

for bank_config in BANK_APPS:
    bank_name = bank_config['bank']
    app_id    = bank_config['app_id']

    print(f"\n{'='*60}")
    print(f"Scraping: {bank_name}")
    print(f"{'='*60}")

    raw_reviews = scrape_reviews(
        app_id=app_id,
        bank_name=bank_name,
        target_count=TARGET_REVIEWS_PER_BANK,
        lang=LANGUAGE,
        country=COUNTRY
    )

    raw_data[bank_name] = raw_reviews
    print(f"  → Collected {len(raw_reviews)} reviews for {bank_name}")

    # Delay between banks
    time.sleep(3)

print("\n✅ Scraping complete for all banks!")
for bank, data in raw_data.items():
    print(f"   {bank}: {len(data)} reviews")

2026-05-17 20:37:13,727 - INFO - Starting scrape for Commercial Bank of Ethiopia (app_id: com.combanketh.mobilebanking)



Scraping: Commercial Bank of Ethiopia


2026-05-17 20:37:15,057 - INFO -   Batch 1: fetched 200 reviews | Total so far: 200
2026-05-17 20:37:18,086 - INFO -   Batch 2: fetched 200 reviews | Total so far: 400
2026-05-17 20:37:21,269 - INFO -   Batch 3: fetched 200 reviews | Total so far: 600
2026-05-17 20:37:23,272 - INFO - ✅ Done scraping Commercial Bank of Ethiopia: 600 total reviews collected


  → Collected 600 reviews for Commercial Bank of Ethiopia


2026-05-17 20:37:26,281 - INFO - Starting scrape for Bank of Abyssinia (app_id: com.boa.boaMobileBanking)


...

  → Collected 600 reviews for Bank of Abyssinia


2026-05-17 20:37:38,943 - INFO - Starting scrape for Dashen Bank (app_id: com.dashen.dashensuperapp)



Scraping: Dashen Bank


2026-05-17 20:37:41,093 - INFO -   Batch 1: fetched 200 reviews | Total so far: 200
2026-05-17 20:37:44,609 - INFO -   Batch 2: fetched 200 reviews | Total so far: 400
2026-05-17 20:37:47,703 - INFO -   Batch 3: fetched 200 reviews | Total so far: 600
2026-05-17 20:37:49,706 - INFO - ✅ Done scraping Dashen Bank: 600 total reviews collected


  → Collected 600 reviews for Dashen Bank

✅ Scraping complete for all banks!
   Commercial Bank of Ethiopia: 600 reviews
   Bank of Abyssinia: 600 reviews
   Dashen Bank: 600 reviews


## 5. Convert Raw Data to DataFrame

We extract only the fields we need and add `bank` and `source` columns.

In [39]:
def parse_reviews_to_dataframe(raw_data_dict, bank_apps_config):
    """
    Convert raw scraped review dicts into a clean pandas DataFrame.

    Parameters:
        raw_data_dict   (dict): {bank_name: [raw_review, ...]}
        bank_apps_config (list): original config list with bank metadata

    Returns:
        pd.DataFrame: combined raw DataFrame with all banks
    """
    frames = []

    # Build a lookup for bank name → app_name
    app_name_lookup = {b['bank']: b['app_name'] for b in bank_apps_config}

    for bank_name, review_list in raw_data_dict.items():
        records = []
        for r in review_list:
            records.append({
                'review_id'   : r.get('reviewId', ''),
                'review'      : r.get('content', ''),
                'rating'      : r.get('score', None),
                'date'        : r.get('at', None),
                'bank'        : bank_name,
                'app_name'    : app_name_lookup.get(bank_name, ''),
                'source'      : 'Google Play',
                'thumbs_up'   : r.get('thumbsUpCount', 0),
                'reply'       : r.get('replyContent', '')
            })
        frames.append(pd.DataFrame(records))
        print(f"   Parsed {len(records)} records for {bank_name}")

    df = pd.concat(frames, ignore_index=True)
    return df


df_raw = parse_reviews_to_dataframe(raw_data, BANK_APPS)

print(f"\n✅ Combined DataFrame shape: {df_raw.shape}")
print(f"   Columns: {list(df_raw.columns)}")
df_raw.head()

   Parsed 600 records for Commercial Bank of Ethiopia
   Parsed 600 records for Bank of Abyssinia
   Parsed 600 records for Dashen Bank

✅ Combined DataFrame shape: (1800, 9)
   Columns: ['review_id', 'review', 'rating', 'date', 'bank', 'app_name', 'source', 'thumbs_up', 'reply']


,review_id,review,rating,date,bank,app_name,source,thumbs_up,reply
0,f11ba9ef-c1a1-4006-9ead-afb585624f63,Good,5,2026-05-16 19:03:11,Commercial Bank of Ethiopia,CBE Mobile,Google Play,0,None
1,5c08f975-fcca-4b2d-8044-25490b83e988,🤙🏼🤙🏼,5,2026-05-16 15:50:50,Commercial Bank of Ethiopia,CBE Mobile,Google Play,0,None
2,c1e25b5d-7e79-4b60-8aa5-bed69a904f62,worst,1,2026-05-16 12:15:55,Commercial Bank of Ethiopia,CBE Mobile,Google Play,0,None
3,eb3cc438-1c10-4e72-8851-3efff6a04135,this app very full,5,2026-05-16 09:17:00,Commercial Bank of Ethiopia,CBE Mobile,Google Play,0,None
4,f8209985-ea16-4f28-bb48-d6a7276f0f08,good apps,4,2026-05-16 07:18:33,Commercial Bank of Ethiopia,CBE Mobile,Google Play,0,None


## 6. Preprocessing

### 6.1 — Check Raw Data Quality

In [40]:
print("=" * 50)
print("RAW DATA QUALITY REPORT")
print("=" * 50)

print(f"\nTotal reviews (raw): {len(df_raw)}")
print(f"\nReviews per bank:")
print(df_raw['bank'].value_counts())

print(f"\nMissing values per column:")
print(df_raw.isnull().sum())

print(f"\nDuplicate review IDs: {df_raw['review_id'].duplicated().sum()}")
print(f"Duplicate review text: {df_raw['review'].duplicated().sum()}")

print(f"\nRating distribution:")
print(df_raw['rating'].value_counts().sort_index())

print(f"\nDate range:")
print(f"  Earliest: {df_raw['date'].min()}")
print(f"  Latest:   {df_raw['date'].max()}")

RAW DATA QUALITY REPORT

Total reviews (raw): 1800

Reviews per bank:
bank
Commercial Bank of Ethiopia    600
Bank of Abyssinia              600
Dashen Bank                    600
Name: count, dtype: int64

Missing values per column:
review_id       0
review          0
rating          0
date            0
bank            0
app_name        0
source          0
thumbs_up       0
reply        1798
dtype: int64

Duplicate review IDs: 0
Duplicate review text: 439

Rating distribution:
rating
1     385
2      61
3      97
4     132
5    1125
Name: count, dtype: int64

Date range:
  Earliest: 2024-11-12 12:45:45
  Latest:   2026-05-16 19:03:11


### 6.2 — Remove Duplicates

In [41]:
before = len(df_raw)

# Drop duplicates by review_id first (exact same submission)
df_clean = df_raw.drop_duplicates(subset='review_id', keep='first')

# Also drop reviews with identical text within the same bank
df_clean = df_clean.drop_duplicates(subset=['review', 'bank'], keep='first')

after = len(df_clean)
print(f"Removed {before - after} duplicate reviews")
print(f"Remaining: {after} reviews")

Removed 352 duplicate reviews
Remaining: 1448 reviews


### 6.3 — Handle Missing Values

In [42]:
before = len(df_clean)

# Drop rows where review text is missing or empty
df_clean = df_clean.dropna(subset=['review'])
df_clean = df_clean[df_clean['review'].str.strip() != '']

# Drop rows where rating is missing
df_clean = df_clean.dropna(subset=['rating'])

after = len(df_clean)
missing_dropped = before - after
missing_pct = (missing_dropped / before) * 100 if before > 0 else 0

print(f"Dropped {missing_dropped} rows with missing review text or rating ({missing_pct:.2f}%)")
print(f"Remaining: {after} reviews")
print(f"Missing data rate: {missing_pct:.2f}% (target: <5%)")

if missing_pct < 5:
    print("✅ Missing data within acceptable threshold (<5%)")
else:
    print("⚠️  Missing data exceeds 5% — investigate further")

Dropped 0 rows with missing review text or rating (0.00%)
Remaining: 1448 reviews
Missing data rate: 0.00% (target: <5%)
✅ Missing data within acceptable threshold (<5%)


### 6.4 — Normalize Dates to YYYY-MM-DD

In [43]:
def normalize_date(date_val):
    """
    Convert various date formats to YYYY-MM-DD string.
    Handles datetime objects, timestamps, and strings.
    """
    if pd.isnull(date_val):
        return None
    try:
        # Already a datetime or pandas Timestamp
        if isinstance(date_val, (datetime, pd.Timestamp)):
            return date_val.strftime('%Y-%m-%d')
        # String format
        return pd.to_datetime(str(date_val)).strftime('%Y-%m-%d')
    except Exception:
        return None


df_clean['date'] = df_clean['date'].apply(normalize_date)

# Check for any dates that couldn't be parsed
null_dates = df_clean['date'].isnull().sum()
print(f"Date normalization complete")
print(f"Null dates after normalization: {null_dates}")
print(f"Sample dates: {df_clean['date'].dropna().head(5).tolist()}")

Date normalization complete
Null dates after normalization: 0
Sample dates: ['2026-05-16', '2026-05-16', '2026-05-16', '2026-05-16', '2026-05-16']


### 6.5 — Clean Review Text

In [44]:
def clean_text(text):
    """
    Basic text cleaning:
    - Strip leading/trailing whitespace
    - Collapse multiple spaces
    - Remove non-printable characters
    """
    if pd.isnull(text):
        return ''
    text = str(text).strip()
    # Collapse multiple whitespace
    text = ' '.join(text.split())
    # Remove non-printable characters
    text = ''.join(ch for ch in text if ch.isprintable())
    return text


df_clean['review'] = df_clean['review'].apply(clean_text)

# Remove reviews that became empty after cleaning
df_clean = df_clean[df_clean['review'].str.len() > 2]

print(f"Text cleaning complete")
print(f"Reviews remaining after text clean: {len(df_clean)}")
print(f"\nSample cleaned reviews:")
for sample in df_clean['review'].head(3).tolist():
    print(f"  • {sample[:100]}")

Text cleaning complete
Reviews remaining after text clean: 1429

Sample cleaned reviews:
  • Good
  • 🤙🏼🤙🏼
  • worst


### 6.6 — Ensure Correct Data Types

In [45]:
# Rating should be integer 1–5
df_clean['rating'] = df_clean['rating'].astype(int)

# Clamp ratings to valid range just in case
df_clean = df_clean[df_clean['rating'].between(1, 5)]

# Reset index after all filtering
df_clean = df_clean.reset_index(drop=True)

print("Data types after normalization:")
print(df_clean.dtypes)
print(f"\nRating range: {df_clean['rating'].min()} – {df_clean['rating'].max()}")

Data types after normalization:
review_id    object
review       object
rating        int32
date         object
bank         object
app_name     object
source       object
thumbs_up     int64
reply        object
dtype: object

Rating range: 1 – 5


## 7. Final Dataset Overview

In [46]:
print("=" * 60)
print("FINAL CLEAN DATASET SUMMARY")
print("=" * 60)

print(f"\nTotal reviews: {len(df_clean)}")

print(f"\nReviews per bank:")
bank_counts = df_clean['bank'].value_counts()
for bank, count in bank_counts.items():
    status = '✅' if count >= 400 else '⚠️ Below target'
    print(f"   {status} {bank}: {count}")

print(f"\nMissing values in final dataset:")
print(df_clean[['review', 'rating', 'date', 'bank', 'source']].isnull().sum())

print(f"\nRating distribution:")
print(df_clean['rating'].value_counts().sort_index())

print(f"\nDate range: {df_clean['date'].min()} → {df_clean['date'].max()}")

print(f"\nSample rows:")
df_clean[['review', 'rating', 'date', 'bank', 'source']].head(5)

FINAL CLEAN DATASET SUMMARY

Total reviews: 1429

Reviews per bank:
   ✅ Dashen Bank: 491
   ✅ Bank of Abyssinia: 490
   ✅ Commercial Bank of Ethiopia: 448

Missing values in final dataset:
review    0
rating    0
date      0
bank      0
source    0
dtype: int64

Rating distribution:
rating
1    372
2     59
3     86
4    103
5    809
Name: count, dtype: int64

Date range: 2024-11-12 → 2026-05-16

Sample rows:


,review,rating,date,bank,source
0,Good,5,2026-05-16,Commercial Bank of Ethiopia,Google Play
1,🤙🏼🤙🏼,5,2026-05-16,Commercial Bank of Ethiopia,Google Play
2,worst,1,2026-05-16,Commercial Bank of Ethiopia,Google Play
3,this app very full,5,2026-05-16,Commercial Bank of Ethiopia,Google Play
4,good apps,4,2026-05-16,Commercial Bank of Ethiopia,Google Play


## 8. Save Clean Dataset to CSV

In [47]:
# Select only the required 5 columns for the final CSV
df_final = df_clean[['review', 'rating', 'date', 'bank', 'source']].copy()

# Create output directory if it doesn't exist
output_dir = '../data/raw'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'reviews_clean.csv')
df_final.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Clean dataset saved to: {output_path}")
print(f"   Shape: {df_final.shape}")
print(f"   File size: {os.path.getsize(output_path) / 1024:.1f} KB")

# Also save the extended version (with review_id for Task 2/3 joins)
df_extended = df_clean[['review', 'rating', 'date', 'bank', 'source', 'review_id', 'thumbs_up']].copy()
df_extended.to_csv(os.path.join(output_dir, 'reviews_extended.csv'), index=False, encoding='utf-8')
print(f"✅ Extended dataset (with review_id) also saved as reviews_extended.csv")

✅ Clean dataset saved to: ../data/raw\reviews_clean.csv
   Shape: (1429, 5)
   File size: 147.8 KB
✅ Extended dataset (with review_id) also saved as reviews_extended.csv


## 9. Quick Validation Check

In [48]:
# Reload and verify the saved CSV
df_verify = pd.read_csv(output_path)

print("=" * 60)
print("CSV VALIDATION CHECK")
print("=" * 60)

# Check 1: Required columns present
required_cols = ['review', 'rating', 'date', 'bank', 'source']
missing_cols = [c for c in required_cols if c not in df_verify.columns]
print(f"\n✅ Required columns present" if not missing_cols else f"❌ Missing columns: {missing_cols}")

# Check 2: Total reviews ≥ 1200
total = len(df_verify)
print(f"{'✅' if total >= 1200 else '⚠️'} Total reviews: {total} (target: 1,200+)")

# Check 3: ≥ 400 per bank
for bank, count in df_verify['bank'].value_counts().items():
    print(f"{'✅' if count >= 400 else '⚠️'} {bank}: {count} reviews (target: 400+)")

# Check 4: Missing data < 5%
total_cells = df_verify[required_cols].size
missing_cells = df_verify[required_cols].isnull().sum().sum()
missing_pct = (missing_cells / total_cells) * 100
print(f"{'✅' if missing_pct < 5 else '⚠️'} Missing data: {missing_pct:.2f}% (target: <5%)")

# Check 5: Date format
sample_date = df_verify['date'].dropna().iloc[0]
try:
    datetime.strptime(sample_date, '%Y-%m-%d')
    print(f"✅ Date format correct: YYYY-MM-DD (sample: {sample_date})")
except:
    print(f"⚠️  Date format issue: {sample_date}")

# Check 6: Rating range
ratings_ok = df_verify['rating'].between(1, 5).all()
print(f"{'✅' if ratings_ok else '⚠️'} All ratings in range 1–5")

# Check 7: Source column
sources_ok = (df_verify['source'] == 'Google Play').all()
print(f"{'✅' if sources_ok else '⚠️'} Source column: 'Google Play'")

print("\n✅ Validation complete — Task 1 is done!")

CSV VALIDATION CHECK

✅ Required columns present
✅ Total reviews: 1429 (target: 1,200+)
✅ Dashen Bank: 491 reviews (target: 400+)
✅ Bank of Abyssinia: 490 reviews (target: 400+)
✅ Commercial Bank of Ethiopia: 448 reviews (target: 400+)
✅ Missing data: 0.00% (target: <5%)
✅ Date format correct: YYYY-MM-DD (sample: 2026-05-16)
✅ All ratings in range 1–5
✅ Source column: 'Google Play'

✅ Validation complete — Task 1 is done!


## 10. Scraping Methodology Summary

### Tool Used
- **Library**: `google-play-scraper` (Python) — a third-party library that interfaces with the Google Play Store's internal API.

### App IDs
| Bank | App ID | App Name |
|------|--------|----------|
| Commercial Bank of Ethiopia | `com.combanketh.mobilebanking` | CBE Mobile |
| Bank of Abyssinia | `com.boa.boaMobileBanking` | BOA Mobile |
| Dashen Bank | `com.dashen.dashensmart` | Dashen Smart |

### Methodology
- Reviews sorted by **newest first** (`Sort.NEWEST`) to capture the most relevant and recent user feedback.
- Pagination via **continuation tokens** to collect 400+ reviews per bank across multiple batches.
- Language: `en` (English), Country: `et` (Ethiopia).
- A **2-second delay** between batches and **3-second delay** between banks was applied to avoid rate limiting.

### Preprocessing Steps
1. **Deduplication**: Removed duplicate entries by `review_id` and by identical `(review, bank)` pairs.
2. **Missing value handling**: Dropped rows with missing `review` text or `rating`; documented the count.
3. **Date normalization**: Converted all dates to `YYYY-MM-DD` format.
4. **Text cleaning**: Stripped whitespace, collapsed multiple spaces, removed non-printable characters.
5. **Type enforcement**: Ensured `rating` is integer and within range 1–5.

### Limitations
- The `google-play-scraper` library depends on the Play Store's unofficial API, which may return fewer results if rate-limited.
- Reviews are only in **English**; many Ethiopian users write in Amharic, so the dataset may under-represent local sentiment.
- The library does not support filtering by exact date range, so the date range depends on what the API returns.
- If fewer than 400 reviews were available for any bank, the date range was not artificially extended — the limitation is documented here.